# ORION anomaly audit

## Scope and authority

This notebook makes adverse, null, missing, discrepant, and boundary outcomes first-class. It is an audit view over committed atlas records, not an automated explanation engine. Explanations below must remain traceable to source receipts; unresolved causes stay unresolved.


## Theory, methodology, and algorithms

For planned-versus-observed coverage, the exact residual is

$$r_n=n_{\mathrm{planned}}-n_{\mathrm{observed}}.$$

For categorical scientific terminals, do not average labels. Preserve the evidence vector

$$E=(e_{\mathrm{execution}},e_{\mathrm{science}},e_{\mathrm{independence}},e_{\mathrm{authority}}),$$

and apply non-escalation componentwise. A favorable scalar coordinate cannot erase a failing component; a missing prospective component stays `CANNOT_CHECK` or not executed.


In [ ]:
from pathlib import Path
import json
import sys


def find_visualization_root(start=Path.cwd()):
    """Find visualization/ whether Jupyter starts at the repo root or notebooks/."""
    start = start.resolve()
    candidates = [start / "visualization", start, *start.parents]
    for candidate in candidates:
        if candidate.name == "visualization" and (candidate / "data" / "derived" / "atlas.json").exists():
            return candidate
        nested = candidate / "visualization"
        if (nested / "data" / "derived" / "atlas.json").exists():
            return nested
    raise FileNotFoundError(
        "Could not find visualization/data/derived/atlas.json. "
        "Build the atlas from the repository root first."
    )


VIS_ROOT = find_visualization_root()
sys.path.insert(0, str(VIS_ROOT / "src"))
ATLAS_PATH = VIS_ROOT / "data" / "derived" / "atlas.json"
atlas = json.loads(ATLAS_PATH.read_text(encoding="utf-8"))


def as_rows(value):
    """Return normalized records without changing their scientific values."""
    if isinstance(value, list):
        return [row for row in value if isinstance(row, dict)]
    if isinstance(value, dict):
        return [row for row in value.values() if isinstance(row, dict)]
    return []


def first(row, *keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


def paper_id(row):
    raw = str(first(row, "paper_id", "paper", "id", default="UNSCOPED"))
    return raw.replace("ORION-", "")


def exact_status(row):
    return str(first(row, "terminal", "status", "result_state", "authority", default="UNSPECIFIED"))


def numeric_value(row):
    value = first(row, "value", "observed", "count", default=None)
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None


paper_states = as_rows(atlas.get("paper_states", []))
metrics_by_paper = atlas.get("metrics", {})
metrics = as_rows(atlas.get("metric_records", []))
anomalies = as_rows(atlas.get("anomalies", []))
sources = as_rows(atlas.get("sources", []))
des_execution = as_rows(atlas.get("des_execution", []))
framework_mechanics = atlas.get("framework_mechanics", {})

print(f"Atlas: {ATLAS_PATH}")
print(
    f"Loaded {len(paper_states)} paper states, {len(metrics)} metrics, "
    f"{len(anomalies)} anomalies, {len(des_execution)} frozen DES rows and "
    f"{len(sources)} sources."
)


In [ ]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.colors import ListedColormap  # noqa: F401 -- used by heatmap notebooks

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 10,
})

STATE_COLORS = {
    "PASS": "#2e7d32",
    "SUPPORTED": "#2e7d32",
    "FAIL": "#c62828",
    "GATE_NOT_MET": "#c62828",
    "CANNOT_CHECK": "#ef6c00",
    "UNKNOWN": "#6a1b9a",
    "NOT_AUTHORITY": "#455a64",
    "NOT_EXECUTED": "#757575",
}


def state_color(text):
    upper = str(text).upper()
    for token, color in STATE_COLORS.items():
        if token in upper:
            return color
    return "#1565c0"


def human_label(value, width=18):
    # Wrap machine identifiers without changing canonical capitalization.
    cleaned = str(value).replace("_", " ").replace(":", " — ")
    return "\n".join(textwrap.wrap(cleaned, width=width, break_long_words=False))



def print_records(rows, fields, limit=30):
    """Small dependency-free table for exact atlas fields."""
    rows = list(rows)
    if not rows:
        print("No records match the current display selectors.")
        return
    widths = {
        field: min(
            48,
            max(len(field), *(len(str(first(row, field, default=""))) for row in rows[:limit])),
        )
        for field in fields
    }
    print(" | ".join(field.ljust(widths[field]) for field in fields))
    print("-+-".join("-" * widths[field] for field in fields))
    for row in rows[:limit]:
        print(" | ".join(str(first(row, field, default=""))[: widths[field]].ljust(widths[field]) for field in fields))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more record(s); change DISPLAY_LIMIT to inspect them.")


## Editable selectors and exact-label filter

Anomaly labels are categorical and are not assigned an unsupported ordinal severity rank. Paper and exact-status filters change only the visible subset. The unfiltered anomaly count is always printed.


In [ ]:
PAPERS = [f"P{i}" for i in range(1, 16)]
STATUS_CONTAINS = None
DISPLAY_LIMIT = 100


## Results: anomaly distribution and paper map

The left plot counts exact stored anomaly labels. The right plot maps each retained record to its paper and exact label; P1 has no retained anomaly record. Counts reflect the atlas inventory, not event rates or comparative paper quality.


In [ ]:
selected_anomalies = [
    row for row in anomalies
    if paper_id(row) in PAPERS
    and (STATUS_CONTAINS is None or STATUS_CONTAINS.upper() in exact_status(row).upper())
]


severity_labels = sorted({str(first(row, "severity", default="UNSPECIFIED")) for row in selected_anomalies})
severity_counts = [sum(str(first(row, "severity", default="UNSPECIFIED")) == label for row in selected_anomalies) for label in severity_labels]
fig, (ax_severity, ax_inventory) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [0.9, 1.2]})
if selected_anomalies:
    y = list(range(len(severity_labels)))
    bars = ax_severity.barh(y, severity_counts, color="#6a1b9a", height=0.68)
    ax_severity.set_yticks(y, [human_label(label, 18) for label in severity_labels])
    ax_severity.invert_yaxis()
    ax_severity.set_title("Retained records by exact anomaly label")
    ax_severity.set_xlabel("anomaly records (count)")
    ax_severity.set_ylabel("exact label")
    ax_severity.set_xlim(0, max(severity_counts) + 0.6)
    ax_severity.set_xticks(range(0, max(severity_counts) + 1))
    for bar, count in zip(bars, severity_counts):
        ax_severity.text(count + 0.08, bar.get_y() + bar.get_height() / 2, str(count), va="center")

    severity_order = severity_labels
    severity_index = {label: index for index, label in enumerate(severity_order)}
    paper_index = {pid: index for index, pid in enumerate(PAPERS)}
    palette = {
        "FAIL": "#c62828",
        "ADVERSE": "#ad1457",
        "CANNOT_CHECK": "#ef6c00",
        "BOUNDARY": "#1565c0",
        "NULL": "#6a1b9a",
        "NOT_AUTHORITY": "#455a64",
        "MIXED": "#5d4037",
    }
    for row in selected_anomalies:
        pid = paper_id(row)
        severity = str(first(row, "severity", default="UNSPECIFIED"))
        ax_inventory.scatter(
            paper_index[pid],
            severity_index[severity],
            s=88,
            marker="o",
            color=palette.get(severity.upper(), "#757575"),
            edgecolor="white",
            linewidth=0.8,
            zorder=3,
        )
    ax_inventory.set_xticks(range(len(PAPERS)), PAPERS)
    ax_inventory.set_yticks(range(len(severity_order)), [human_label(label, 18) for label in severity_order])
    ax_inventory.set_xlim(-0.6, len(PAPERS) - 0.4)
    ax_inventory.set_ylim(-0.6, len(severity_order) - 0.4)
    ax_inventory.invert_yaxis()
    ax_inventory.set_title("Which anomaly label is retained for each paper")
    ax_inventory.set_xlabel("paper (P1 has no retained anomaly record)")
    ax_inventory.set_ylabel("exact anomaly label")
    ax_inventory.grid(axis="x", alpha=0.18)
else:
    for ax in (ax_severity, ax_inventory):
        ax.text(0.5, 0.5, "No anomalies match the display selectors", ha="center", va="center")
        ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
print_records(
    selected_anomalies,
    ["paper_id", "anomaly_id", "severity", "status", "summary", "observed", "expected", "explanation", "source_ids"],
    DISPLAY_LIMIT,
)


## Discussion: mandatory retained findings

1. **P2:** overall `FAIL` despite nDCG; a favorable coordinate cannot replace the registered terminal.
2. **P5:** requested `glm-5.2`, served `glm-5.3`; requested-condition identity was not met.
3. **P7:** 738 planned, 736 observed; the run is invalid and the coverage residual is 2.
4. **P9:** digits D-A is `CANNOT_CHECK`; the replay-revival receipt keeps the old failure terminal, records archive-matched agreement, and leaves the scientific successor unexecuted.
5. **P10:** prospective experiment not executed; no empirical promotion is available.
6. **P11:** terminal `GATE_NOT_MET`, despite available query/support-count rows.
7. **P12:** `forward_time_deployability=CANNOT_CHECK`; the registered public-data stop/go campaign is not executed.
8. **P13:** bounded finite result plus the separately registered historical adverse terminal; neither may erase the other.
9. **P14:** the 67-packet pilot analytics are `NOT_AUTHORITY`; internally authored cases are not external validation.
10. **P15:** full key compromise produced 0 signature detections and 6 false promotions; verification cannot establish custody, fact truth, or scientific authority.

Use `visualization/reports/ANOMALY_AUDIT.md` for the explanation and next-discriminator table.

## Claim ceiling

An anomaly plot can expose contradictions, missingness, and boundary cases. It cannot repair them. Until the relevant frozen discriminator is executed and bound, adverse statuses remain adverse and unresolved statuses remain `CANNOT_CHECK`.
